<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Deep-Learning/08-sequence-models-state-space-models.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Deep Learning guideline](Deep-Learning.html)

## **Sequence Models and State-Space Models** {#sequence-models-state-space-models}

Chapter 07 encoded two-dimensional locality with shared spatial kernels. Sequential data introduces a different geometry: observations arrive in an order, sequence lengths vary, and the meaning of one element may depend on a distant history. A sequence model must decide **what state to preserve, how information moves across positions, and which computations can be parallelized**.

This chapter uses one real sequence throughout: the [OpenSLR SLR1 YESNO corpus](https://openslr.org/1/), also documented by the official [torchaudio `YESNO` dataset](https://docs.pytorch.org/audio/main/generated/torchaudio.datasets.YESNO.html). It contains 60 recordings from one male speaker, sampled at 8 kHz. Each filename gives an eight-word Hebrew yes/no sequence, for example `1_0_0_0_0_0_1_1.wav`. OpenSLR reports no formal license and states that the data is free to use for any purpose. The small archive is stored locally so every example remains offline and reproducible.

Each waveform is transformed into log-magnitude short-time Fourier transform (STFT) frames. If $x[n]$ is audio, window $w[n]$, frame index $t$, and frequency bin $k$, then

$$
X[t,k]=\sum_{n=0}^{N-1}x[n+tH]w[n]e^{-j2\pi kn/N},
\qquad
z_t[k]=\log(1+|X[t,k]|).
$$

$N$ is the FFT size and $H$ the hop length. The model therefore receives a variable-length sequence $Z\in\mathbb{R}^{T\times F}$ rather than raw samples. Training-set statistics normalize every frequency channel. The main compact classification task predicts the **first spoken word**; the CTC section uses all eight labels.

### **Sequential Data and Temporal Dependence** {#sequential-data-temporal-dependence}

A sequence $x_{1:T}=(x_1,\ldots,x_T)$ is not an unordered set. Permuting frames changes phonetic evolution even if the same values remain. Three common output structures are:

- **sequence-to-label:** one output for an entire utterance, such as intent or speaker identity;
- **sequence-to-sequence with alignment:** one output per frame or token, such as tagging;
- **sequence-to-sequence without known alignment:** a shorter target sequence, handled by attention or CTC.

Variable lengths require masks or packed representations. Padding is an implementation value, not an observation. If padded frames update hidden states, affect normalization, or enter a pooled average, the model learns from batch formatting. A binary mask $M_{b,t}=1[t<L_b]$ distinguishes valid positions for sample $b$ with length $L_b$.

Temporal dependence also has direction. A streaming recognizer may only use $x_{1:t}$, while an offline transcription model can use past and future context. Bidirectionality can improve offline accuracy but violates a causal latency contract. The available context must therefore be specified as part of the task, not only the architecture.

<details>
<summary><strong>PyTorch: load and normalize the shared YESNO sequences</strong></summary>

```python
import io
import math
import random
import tarfile
import wave
from pathlib import Path

import numpy as np
import torch
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from torch.utils.data import DataLoader, Dataset

torch.set_num_threads(1)


def seed_everything(seed=808):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def read_pcm_wave(payload):
    with wave.open(io.BytesIO(payload), "rb") as stream:
        assert stream.getnchannels() == 1 and stream.getsampwidth() == 2
        sample_rate = stream.getframerate()
        samples = np.frombuffer(stream.readframes(stream.getnframes()), dtype="<i2").copy()
    return torch.tensor(samples, dtype=torch.float32) / 32768.0, sample_rate


archive_candidates = [
    Path("assets/data/waves_yesno.tar.gz"),
    Path("ipynb/Deep-Learning/assets/data/waves_yesno.tar.gz"),
]
archive = next(path for path in archive_candidates if path.exists())
records = []
window = torch.hann_window(256)
with tarfile.open(archive, "r:gz") as bundle:
    members = sorted(
        (m for m in bundle.getmembers() if m.isfile() and m.name.endswith(".wav")),
        key=lambda member: member.name,
    )
    for member in members:
        waveform, sample_rate = read_pcm_wave(bundle.extractfile(member).read())
        spectrum = torch.stft(
            waveform, n_fft=256, hop_length=160, win_length=256,
            window=window, return_complex=True,
        ).abs().transpose(0, 1)
        features = torch.log1p(spectrum)
        labels = torch.tensor([int(value) for value in Path(member.name).stem.split("_")])
        records.append({"name": member.name, "features": features, "labels": labels})

all_idx = np.arange(len(records))
first_labels = np.array([int(record["labels"][0]) for record in records])
train_idx, holdout_idx = train_test_split(
    all_idx, test_size=0.30, random_state=808, stratify=first_labels
)
val_idx, test_idx = train_test_split(
    holdout_idx, test_size=0.50, random_state=808, stratify=first_labels[holdout_idx]
)

# Fit feature normalization on training frames only.
training_frames = torch.cat([records[i]["features"] for i in train_idx], dim=0)
feature_mean = training_frames.mean(0)
feature_std = training_frames.std(0).clamp_min(1e-5)
for record in records:
    record["features"] = (record["features"] - feature_mean) / feature_std


class YesNoDataset(Dataset):
    def __init__(self, indices):
        self.indices = list(map(int, indices))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        return records[self.indices[index]]


def collate_yesno(batch):
    sequences = [item["features"] for item in batch]
    lengths = torch.tensor([len(sequence) for sequence in sequences])
    padded = pad_sequence(sequences, batch_first=True)
    labels = torch.stack([item["labels"] for item in batch])
    mask = torch.arange(padded.shape[1]).unsqueeze(0) < lengths.unsqueeze(1)
    return padded, lengths, mask, labels


def yesno_loader(indices, shuffle=False, seed=808, batch_size=8):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        YesNoDataset(indices), batch_size=batch_size, shuffle=shuffle,
        generator=generator, collate_fn=collate_yesno,
    )


sample_batch = next(iter(yesno_loader(train_idx)))
padded, lengths, valid_mask, word_labels = sample_batch
assert len(records) == 60 and sample_rate == 8000
assert padded.shape[0] == 8 and padded.shape[2] == 129
assert valid_mask.sum(1).equal(lengths)
assert word_labels.shape == (8, 8)
assert set(train_idx).isdisjoint(test_idx)
print({"recordings": len(records), "split": (len(train_idx), len(val_idx), len(test_idx)),
       "batch shape": tuple(padded.shape), "length range": (int(lengths.min()), int(lengths.max()))})
```

</details>

The split and normalized features are reused below. Because the corpus contains one speaker and only 60 utterances, results demonstrate mechanisms rather than broad speech-recognition generalization. Claims about speakers, accents, languages, or noise conditions require a larger grouped dataset.

### **Recurrent Neural Networks** {#recurrent-neural-networks}

A recurrent neural network compresses the prefix $x_{1:t}$ into hidden state $h_t$:

$$
a_t=W_{xh}x_t+W_{hh}h_{t-1}+b_h,
\qquad
h_t=\phi(a_t),
\qquad
o_t=W_{hy}h_t+b_y.
$$

$x_t\in\mathbb{R}^{F}$ is one acoustic frame, $h_t\in\mathbb{R}^{H}$ is a learned summary, and the same matrices are reused at every time step. Weight sharing lets the model process arbitrary lengths and expresses time-stationary transition rules. It also creates a sequential dependency: $h_t$ cannot be computed before $h_{t-1}$.

For sequence classification, use the last **valid** hidden state, not the final padded position. `pack_padded_sequence` lets PyTorch skip padded updates. Alternatively, gather `output[b, L_b-1]` from an unpacked tensor with a correct mask. For framewise tasks, keep every valid output and mask the loss.

The hidden state is a bottleneck. It must preserve information useful to future predictions while continually incorporating new evidence. A vanilla `tanh` RNN can model short dependencies efficiently, but repeated Jacobian products make long-memory optimization difficult. Gated cells change this state-update path.

![An RNN cell is unfolded across time, revealing shared parameters and a hidden-state dependency between successive inputs.](assets/dl08-unfolded-rnn.svg){fig-align="center" width="74%" fig-alt="Recurrent neural network unfolded across time with shared cell parameters and hidden states."}

*Image source: [Dive into Deep Learning, Recurrent Neural Networks](https://d2l.ai/chapter_recurrent-neural-networks/rnn.html), CC BY-SA 4.0.*

<details>
<summary><strong>PyTorch: train a vanilla RNN on the first YESNO word</strong></summary>

```python
class RecurrentFirstWord(nn.Module):
    def __init__(self, cell="rnn", input_size=129, hidden_size=32, bidirectional=False):
        super().__init__()
        cells = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}
        self.encoder = cells[cell](
            input_size, hidden_size, batch_first=True,
            nonlinearity="tanh" if cell == "rnn" else None,
            bidirectional=bidirectional,
        ) if cell == "rnn" else cells[cell](input_size, hidden_size, batch_first=True,
                                             bidirectional=bidirectional)
        directions = 2 if bidirectional else 1
        self.head = nn.Linear(hidden_size * directions, 2)

    def forward(self, x, lengths):
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, state = self.encoder(packed)
        hidden = state[0] if isinstance(state, tuple) else state
        if self.encoder.bidirectional:
            representation = torch.cat([hidden[-2], hidden[-1]], dim=-1)
        else:
            representation = hidden[-1]
        return self.head(representation)


@torch.no_grad()
def sequence_accuracy(model, indices):
    model.eval()
    correct = total = 0
    for xb, lens, _, labels in yesno_loader(indices, batch_size=8):
        prediction = model(xb, lens).argmax(1)
        correct += int((prediction == labels[:, 0]).sum())
        total += len(labels)
    return correct / total


def fit_sequence_classifier(model, epochs=12, seed=809):
    seed_everything(seed)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-3)
    for _ in range(epochs):
        model.train()
        for xb, lens, _, labels in yesno_loader(train_idx, shuffle=True, seed=seed):
            optimizer.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(xb, lens), labels[:, 0])
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
    return {"validation": sequence_accuracy(model, val_idx),
            "test": sequence_accuracy(model, test_idx)}


seed_everything(809)
vanilla_rnn = RecurrentFirstWord("rnn")
rnn_result = fit_sequence_classifier(vanilla_rnn, epochs=10, seed=809)
assert vanilla_rnn(padded, lengths).shape == (8, 2)
assert all(0 <= value <= 1 for value in rnn_result.values())
print(rnn_result)
```

</details>

The score is intentionally secondary to the state interface. With nine validation and nine test recordings from one speaker, one mistake changes accuracy by eleven percentage points. The correct conclusion is that the implementation handles real variable-length audio, not that one small RNN has established a speech benchmark.

### **Backpropagation Through Time** {#backpropagation-through-time}

Backpropagation through time (BPTT) unfolds a recurrent computation into a depth-$T$ graph and applies reverse-mode differentiation. If total loss is $L=\sum_t\ell_t$, then a shared recurrent matrix receives contributions from every use:

$$
\frac{\partial L}{\partial W_{hh}}
=\sum_{t=1}^{T}
\frac{\partial L}{\partial h_t}
\frac{\partial h_t}{\partial W_{hh}}.
$$

Influence from time $t$ back to time $k$ contains a product of Jacobians,

$$
\frac{\partial h_t}{\partial h_k}
=\prod_{j=k+1}^{t}
\frac{\partial h_j}{\partial h_{j-1}}.
$$

If typical singular values are below one, gradients vanish exponentially; above one, they can explode. `tanh` and sigmoid derivatives further contract saturated paths. Gradient clipping limits an explosion after it occurs but does not restore vanished information.

Full BPTT stores activations for all steps and has memory proportional to sequence length. **Truncated BPTT** processes windows, carries the numerical hidden state forward, and detaches it between windows. It reduces memory and update latency but treats dependencies crossing the truncation boundary as constants. The truncation length is therefore a modeling approximation, not merely a systems setting.

<details>
<summary><strong>PyTorch: trace gradient decay across one real acoustic sequence</strong></summary>

```python
# Use the first 50 normalized acoustic frames and retain each hidden state gradient.
acoustic_prefix = records[int(train_idx[0])]["features"][:50]
input_projection = nn.Linear(129, 24, bias=False)
recurrent = nn.Linear(24, 24, bias=False)
with torch.no_grad():
    recurrent.weight.mul_(0.35)  # Create a contractive transition for a visible diagnostic.

hidden = torch.zeros(24)
hidden_states = []
for frame in acoustic_prefix:
    hidden = torch.tanh(input_projection(frame) + recurrent(hidden))
    hidden.retain_grad()
    hidden_states.append(hidden)

loss = hidden_states[-1].pow(2).mean()
loss.backward()
gradient_norms = torch.tensor([state.grad.norm().item() for state in hidden_states])

assert len(gradient_norms) == 50
assert torch.isfinite(gradient_norms).all()
assert gradient_norms[-1] > gradient_norms[0]
print({"first-step gradient": gradient_norms[0].item(),
       "last-step gradient": gradient_norms[-1].item(),
       "ratio": (gradient_norms[-1] / gradient_norms[0].clamp_min(1e-30)).item()})
```

</details>

The deliberately contractive matrix makes one failure mode easy to observe; a trained network can show mixed growth and decay across directions. Diagnose gradients by layer and time, and distinguish optimization failure from a task that simply does not require distant context.

### **Long Short-Term Memory** {#long-short-term-memory}

An LSTM introduces cell state $c_t$ with an additive update:

$$
\begin{aligned}
i_t&=\sigma(W_i x_t+U_i h_{t-1}+b_i),\\
f_t&=\sigma(W_f x_t+U_f h_{t-1}+b_f),\\
g_t&=\tanh(W_g x_t+U_g h_{t-1}+b_g),\\
o_t&=\sigma(W_o x_t+U_o h_{t-1}+b_o),\\
c_t&=f_t\odot c_{t-1}+i_t\odot g_t,\\
h_t&=o_t\odot\tanh(c_t).
\end{aligned}
$$

The forget gate $f_t$ preserves or removes old memory, input gate $i_t$ controls new content, candidate $g_t$ proposes that content, and output gate $o_t$ exposes selected memory. The direct derivative $\partial c_t/\partial c_{t-1}=f_t$ provides a tunable route for gradients. When $f_t$ remains near one, memory can persist much longer than through repeated unconstrained `tanh` transitions.

Gating does not eliminate every long-range problem. A saturated gate learns slowly, cell values can drift, and each step requires four affine projections. Long sequences remain sequential in training and inference. LSTMs are attractive when data is moderate, streaming state matters, and the recurrence's fixed memory is more useful than all-pairs interaction.

![The LSTM cell separates persistent cell memory from the exposed hidden state using input, forget, and output gates.](assets/dl08-lstm-memory.svg){fig-align="center" width="76%" fig-alt="LSTM cell diagram with cell state, hidden state, and three gates."}

*Image source: local educational redraw based on the standard LSTM equations introduced by [Hochreiter and Schmidhuber (1997)](https://www.bioinf.jku.at/publications/older/2604.pdf).*

<details>
<summary><strong>PyTorch: train an LSTM under the same YESNO protocol</strong></summary>

```python
seed_everything(810)
lstm_classifier = RecurrentFirstWord("lstm")
lstm_result = fit_sequence_classifier(lstm_classifier, epochs=12, seed=810)

parameter_count = lambda model: sum(parameter.numel() for parameter in model.parameters())
expected_lstm_core = 4 * 32 * (129 + 32 + 2)  # weights plus two bias vectors per gate in PyTorch
actual_lstm_core = sum(p.numel() for p in lstm_classifier.encoder.parameters())

assert actual_lstm_core == expected_lstm_core
assert lstm_classifier(padded, lengths).shape == (8, 2)
print({"core parameters": actual_lstm_core, "validation/test": lstm_result})
```

</details>

The four-gate parameter count is part of the trade-off. Compare models at both matched hidden width and matched parameter budget; those are different experiments and can lead to different conclusions.

### **Gated Recurrent Units** {#gated-recurrent-units}

A GRU merges cell and hidden state and usually uses three affine gate groups:

$$
z_t=\sigma(W_zx_t+U_zh_{t-1}),
\qquad
r_t=\sigma(W_rx_t+U_rh_{t-1}),
$$

$$
\widetilde h_t=\tanh(W_hx_t+U_h(r_t\odot h_{t-1})),
\qquad
h_t=(1-z_t)\odot\widetilde h_t+z_t\odot h_{t-1}.
$$

The update gate $z_t$ interpolates between preserving old state and accepting a candidate; the reset gate $r_t$ controls how much old state contributes to the candidate. Some sources reverse the coefficients attached to $z_t$, so interpret a GRU from its equation or implementation rather than the gate name alone.

GRUs typically use fewer parameters and less compute per step than LSTMs at the same hidden width. LSTMs offer a separately controlled cell state; GRUs offer a simpler state interface. Neither dominates across all tasks. Sequence length, data size, latency, and hyperparameter budget often matter more than the nominal cell family.

<details>
<summary><strong>PyTorch: compare GRU and LSTM at matched hidden width</strong></summary>

```python
seed_everything(811)
gru_classifier = RecurrentFirstWord("gru")
gru_result = fit_sequence_classifier(gru_classifier, epochs=12, seed=811)

gru_core = sum(p.numel() for p in gru_classifier.encoder.parameters())
lstm_core = sum(p.numel() for p in lstm_classifier.encoder.parameters())
comparison_batch = padded[:4]
comparison_lengths = lengths[:4]

assert gru_classifier(comparison_batch, comparison_lengths).shape == (4, 2)
assert gru_core < lstm_core
print({"GRU core parameters": gru_core, "LSTM core parameters": lstm_core,
       "GRU validation/test": gru_result})
```

</details>

Because the runs use different initializations and only a tiny validation set, their scores do not establish a cell ranking. The reliable observation is architectural: GRU uses three gate groups and one recurrent state, while LSTM uses four gate groups and separates cell from hidden state.

### **Bidirectional and Encoder-Decoder Recurrent Models** {#bidirectional-encoder-decoder-recurrent-models}

A bidirectional recurrent model computes

$$
\overrightarrow h_t=F(x_t,\overrightarrow h_{t-1}),
\qquad
\overleftarrow h_t=B(x_t,\overleftarrow h_{t+1}),
\qquad
h_t=[\overrightarrow h_t;\overleftarrow h_t].
$$

Each position receives past and future context. This is useful for offline tagging and transcription, but it cannot produce a final backward state until the utterance ends. Chunked bidirectional systems trade future context for bounded latency; a strict streaming system remains causal.

An encoder-decoder separates input representation from output generation. The encoder maps acoustic frames to hidden states; the decoder predicts output tokens autoregressively,

$$
p(y_{1:U}\mid x_{1:T})=\prod_{u=1}^{U}p(y_u\mid y_{<u},x_{1:T}).
$$

A fixed final encoder state creates a bottleneck. Attention lets each decoder step retrieve a weighted combination of all encoder states, while CTC removes the autoregressive decoder under a monotonic conditional-independence assumption. Teacher forcing supplies the true previous token during training; inference supplies the model's own token, creating exposure bias.

<details>
<summary><strong>PyTorch: expose bidirectional states and one teacher-forced decoder step</strong></summary>

```python
bi_encoder = nn.GRU(129, 24, batch_first=True, bidirectional=True)
packed = pack_padded_sequence(padded, lengths.cpu(), batch_first=True, enforce_sorted=False)
packed_output, bi_state = bi_encoder(packed)
encoder_memory, _ = torch.nn.utils.rnn.pad_packed_sequence(packed_output, batch_first=True)

token_embedding = nn.Embedding(3, 16)  # 0/1 words plus start token 2
decoder = nn.GRU(input_size=16 + 48, hidden_size=48, batch_first=True)
output_head = nn.Linear(48, 2)

start = torch.full((padded.shape[0], 1), 2, dtype=torch.long)
teacher_inputs = torch.cat([start, word_labels[:, :-1]], dim=1)
context = torch.cat([bi_state[-2], bi_state[-1]], dim=-1).unsqueeze(1).expand(-1, 8, -1)
decoder_input = torch.cat([token_embedding(teacher_inputs), context], dim=-1)
decoder_output, _ = decoder(decoder_input)
teacher_forced_logits = output_head(decoder_output)

assert encoder_memory.shape[:2] == padded.shape[:2]
assert encoder_memory.shape[2] == 48
assert teacher_forced_logits.shape == (8, 8, 2)
print({"encoder memory": tuple(encoder_memory.shape),
       "decoder logits": tuple(teacher_forced_logits.shape)})
```

</details>

The example uses a repeated final context to reveal the bottleneck explicitly. An attention decoder would compute a different context at each output step. In either case, the data-access contract decides whether bidirectional encoding is valid.

### **Temporal Convolution and Connectionist Temporal Classification** {#temporal-convolution-ctc}

A temporal convolutional network (TCN) applies one-dimensional convolutions over time. Dilated causal layers can expand context exponentially: for kernel size $K$ and dilations $1,2,4,\ldots,2^{L-1}$, the receptive field is

$$
R=1+(K-1)\sum_{l=0}^{L-1}2^l
=1+(K-1)(2^L-1).
$$

Unlike recurrence, every position in one convolutional layer can be computed in parallel. A causal TCN pads only the past; a symmetric TCN uses future context. Its memory is finite unless depth or dilation grows, and poorly chosen dilation patterns can miss local interactions.

Connectionist Temporal Classification (CTC) trains when the input has $T$ frames, the target has $U\le T$ symbols, and frame-level alignment is unknown. Add blank symbol $\varnothing$. A path $\pi\in(\mathcal{V}\cup\{\varnothing\})^T$ collapses by removing blanks and repeated adjacent symbols. The sequence probability sums all valid paths:

$$
p(y\mid x)=\sum_{\pi:\mathcal{B}(\pi)=y}\prod_{t=1}^{T}p(\pi_t\mid x).
$$

Dynamic programming computes this sum without enumerating paths. CTC assumes frame predictions are conditionally independent given the encoder and enforces monotonic order. It cannot directly model arbitrary output reordering, and repeated labels require blank separation.

![CTC maps many frame-level paths containing blanks and repeats to the same shorter label sequence.](assets/dl08-ctc-alignment.svg){fig-align="center" width="76%" fig-alt="CTC alignment diagram showing frame paths collapsing to a shorter yes-no label sequence."}

*Image source: local educational diagram derived from the CTC collapse operator in [Graves et al. (2006)](https://www.cs.toronto.edu/~graves/icml_2006.pdf).*

<details>
<summary><strong>PyTorch: compare a TCN encoder and CTC objective on YESNO</strong></summary>

```python
class TemporalBlock(nn.Module):
    def __init__(self, channels, dilation):
        super().__init__()
        self.dilation = dilation
        self.conv = nn.Conv1d(channels, channels, 3, dilation=dilation)

    def forward(self, x):
        # Left-only padding makes the output causal and keeps its length.
        x = F.pad(x, (2 * self.dilation, 0))
        return F.relu(self.conv(x))


projected = nn.Conv1d(129, 32, 1)(padded.transpose(1, 2))
tcn = nn.Sequential(TemporalBlock(32, 1), TemporalBlock(32, 2), TemporalBlock(32, 4))
tcn_features = tcn(projected).transpose(1, 2)
ctc_head = nn.Linear(32, 3)  # labels 0/1 and blank index 2
log_probabilities = ctc_head(tcn_features).log_softmax(-1).transpose(0, 1)

flat_targets = word_labels.flatten()
target_lengths = torch.full((len(word_labels),), 8, dtype=torch.long)
ctc_loss = F.ctc_loss(
    log_probabilities, flat_targets, lengths, target_lengths,
    blank=2, zero_infinity=True,
)

assert tcn_features.shape[:2] == padded.shape[:2]
assert log_probabilities.shape == (padded.shape[1], padded.shape[0], 3)
assert torch.isfinite(ctc_loss)
assert 1 + (3 - 1) * (1 + 2 + 4) == 15
print({"TCN receptive field": 15, "CTC loss": ctc_loss.item()})
```

</details>

The untrained loss validates alignment plumbing only. A full system would train the encoder, decode with greedy collapse or beam search, and report sequence error on speakers not used for fitting.

### **Long-Range Dependency and Parallelism Limits** {#long-range-dependency-parallelism-limits}

Sequence architectures trade off state size, dependency path length, training parallelism, and inference memory.

- In an RNN, information from position 1 reaches position $T$ through $T-1$ transitions. Training is sequential across time, but streaming inference stores only fixed-size state.
- In a dilated TCN, path length can grow logarithmically with covered context, and training is parallel within each layer. Streaming requires a cache of recent layer activations.
- In full self-attention, any two positions interact in one layer, but the score matrix uses $O(T^2)$ memory/compute. Autoregressive inference caches keys and values whose memory grows with context length.
- In a state-space model, a recurrent scan can use fixed-size state for streaming, while a time-invariant form can be evaluated as a parallel convolution during training.

These asymptotic statements do not directly predict wall-clock speed. Kernel fusion, sequence length, hidden width, batch size, memory bandwidth, and accelerator utilization matter. For the YESNO STFT sequences, $T$ is only a few hundred; for long audio, genomic data, or high-rate sensors, quadratic attention becomes much more consequential.

The modeling question is equally important. A long nominal context does not prove long-range information is retained or used. Perturbation tests, gradient-based attribution, controlled retrieval tasks, and performance versus context truncation reveal functional context.

<details>
<summary><strong>Python: compare dependency and memory scaling for the observed lengths</strong></summary>

```python
observed_length = int(lengths.max())
hidden_width = 64
layers = 8
kernel_size = 3
tcn_receptive_field = 1 + (kernel_size - 1) * (2 ** layers - 1)

scaling = {
    "RNN transition path": observed_length - 1,
    "dilated TCN receptive field": tcn_receptive_field,
    "attention score elements per head": observed_length ** 2,
    "streaming recurrent state elements": hidden_width,
    "autoregressive KV elements per layer (one K and one V)": 2 * observed_length * hidden_width,
}

assert tcn_receptive_field >= observed_length
assert scaling["attention score elements per head"] > scaling["RNN transition path"]
print(scaling)
```

</details>

The arithmetic uses a real batch length but remains a resource model, not a benchmark. A benchmark must include warmup, synchronized timing, target precision, and peak memory on the actual hardware.

### **Structured State-Space Models and S4** {#structured-state-space-models-s4}

A continuous-time linear state-space model (SSM) is

$$
\frac{d h(t)}{dt}=Ah(t)+Bx(t),
\qquad
y(t)=Ch(t)+Dx(t),
$$

where state $h(t)\in\mathbb{R}^{N}$ compresses history. After discretization with step $\Delta$,

$$
h_k=\overline A h_{k-1}+\overline Bx_k,
\qquad
y_k=Ch_k+Dx_k.
$$

For zero-order hold, $\overline A=e^{\Delta A}$ and $\overline B=A^{-1}(e^{\Delta A}-I)B$ when $A$ is invertible. A simpler Euler approximation uses $\overline A=I+\Delta A$ and $\overline B=\Delta B$, but can be unstable for a large step.

When $A,B,C$ are time invariant, unrolling yields a convolution

$$
y_k=\sum_{j=0}^{k}K_{k-j}x_j,
\qquad
K_i=C\overline A^i\overline B,
$$

plus the direct $Dx_k$ term. This **recurrent-convolutional duality** permits fixed-state streaming recurrence and parallel full-sequence convolution.

S4 makes long state dimensions computationally practical by imposing structured parameterizations on $A$ and exploiting efficient kernel generation. Its HiPPO-inspired initialization is designed to preserve information about a continuous history rather than using an arbitrary transition. The important point is not that every SSM is S4: S4 combines a state-space view, a particular long-memory initialization, and structured numerical algorithms.

![A linear time-invariant state-space recurrence can be unrolled into a one-dimensional convolution kernel.](assets/dl08-ssm-duality.svg){fig-align="center" width="76%" fig-alt="Diagram comparing recurrent state-space scanning with an equivalent convolution kernel."}

*Image source: local educational redraw based on [Gu, Goel, and Ré, Efficiently Modeling Long Sequences with Structured State Spaces](https://arxiv.org/abs/2111.00396).*

<details>
<summary><strong>PyTorch: verify recurrence-convolution duality on YESNO energy</strong></summary>

```python
# Collapse the first utterance's spectrum to one scalar energy signal per frame.
u = records[int(train_idx[0])]["features"][:80].pow(2).mean(-1)
state_size = 12
continuous_A = -torch.exp(torch.linspace(-2.0, 0.5, state_size))
delta = 0.05
A_bar = torch.exp(delta * continuous_A)
B_bar = torch.randn(state_size) * 0.05
C = torch.randn(state_size)

state = torch.zeros(state_size)
recurrent_output = []
for value in u:
    state = A_bar * state + B_bar * value
    recurrent_output.append(C @ state)
recurrent_output = torch.stack(recurrent_output)

kernel = torch.stack([C @ (A_bar.pow(lag) * B_bar) for lag in range(len(u))])
convolution_output = torch.stack([
    (kernel[:time + 1] * u[:time + 1].flip(0)).sum()
    for time in range(len(u))
])

assert torch.allclose(recurrent_output, convolution_output, atol=1e-5)
assert A_bar.abs().max() < 1  # Stable decaying modes for this diagonal example.
print({"sequence length": len(u), "state size": state_size,
       "maximum duality error": (recurrent_output - convolution_output).abs().max().item()})
```

</details>

The diagonal model exposes the algebra but omits S4's structured matrix machinery and fast kernel algorithm. It is a Tier-2 mechanism implementation: enough to verify the dual view without pretending to reproduce an optimized S4 library.

### **Mamba and Selective State-Space Models** {#mamba-selective-state-space-models}

Linear time-invariant SSMs apply the same dynamics at every position. This is efficient but limits **content-based reasoning**: the model cannot directly decide that one input should be stored while another should be ignored. Mamba makes key SSM quantities input dependent. In a simplified selective update,

$$
\Delta_t=\operatorname{softplus}(W_\Delta x_t),
\quad B_t=W_Bx_t,
\quad C_t=W_Cx_t,
$$

$$
h_t=e^{\Delta_t A}\odot h_{t-1}+\Delta_t B_t\odot u_t,
\qquad
y_t=C_t^\top h_t+D u_t.
$$

$A$ remains a learned stable base dynamic, while $\Delta_t$, $B_t$, and $C_t$ let content control timescale, writing, and reading. A large decay can reset memory; a slow mode can preserve it; input-dependent read vectors expose different state components.

Selectivity breaks the fixed convolution kernel used by linear time-invariant SSMs. Mamba recovers efficiency with a hardware-aware parallel selective scan, plus local convolution and gated projections in the surrounding block. The model remains linear in sequence length in its scan, but practical speed depends on fused kernels, state width, precision, and memory traffic.

Mamba is not simply “attention with linear complexity.” Attention explicitly compares token pairs and can retrieve a particular earlier representation. A selective SSM compresses history into finite state. This can be excellent for long streaming data but may be less direct for exact associative recall unless the state and learned dynamics support it.

![Selective state-space dynamics let input content control how strongly information is written, retained, and read at each step.](assets/dl08-selective-ssm.svg){fig-align="center" width="76%" fig-alt="Selective state-space model diagram showing input-dependent step, write, and read parameters."}

*Image source: local educational redraw based on [Gu and Dao, Mamba: Linear-Time Sequence Modeling with Selective State Spaces](https://arxiv.org/abs/2312.00752).*

<details>
<summary><strong>PyTorch: run a simplified selective scan on acoustic frames</strong></summary>

```python
class SelectiveAcousticSSM(nn.Module):
    def __init__(self, input_size=129, state_size=16):
        super().__init__()
        self.input_projection = nn.Linear(input_size, 1)
        self.delta_projection = nn.Linear(input_size, state_size)
        self.write_projection = nn.Linear(input_size, state_size)
        self.read_projection = nn.Linear(input_size, state_size)
        self.log_decay = nn.Parameter(torch.linspace(-2.0, 0.0, state_size))
        self.direct = nn.Parameter(torch.tensor(0.0))

    def forward(self, x, mask):
        batch, time, _ = x.shape
        state = x.new_zeros(batch, self.log_decay.numel())
        outputs = []
        A = -torch.exp(self.log_decay)
        for step in range(time):
            frame = x[:, step]
            u = self.input_projection(frame)
            delta = F.softplus(self.delta_projection(frame))
            write = torch.tanh(self.write_projection(frame))
            read = torch.tanh(self.read_projection(frame))
            candidate = torch.exp(delta * A) * state + delta * write * u
            active = mask[:, step].unsqueeze(1)
            state = torch.where(active, candidate, state)
            outputs.append((read * state).sum(-1, keepdim=True) + self.direct * u)
        return torch.stack(outputs, dim=1)


selective_ssm = SelectiveAcousticSSM()
selective_output = selective_ssm(padded[:4], valid_mask[:4])

# Padded steps leave state unchanged internally; outputs retain a regular tensor shape.
assert selective_output.shape == (4, padded.shape[1], 1)
assert torch.isfinite(selective_output).all()
loss = selective_output[valid_mask[:4]].pow(2).mean()
loss.backward()
assert selective_ssm.log_decay.grad is not None
print({"output": tuple(selective_output.shape), "loss": loss.item()})
```

</details>

The Python loop is intentionally transparent and slow. A production Mamba block uses expanded channels, local causal convolution, gating, normalization, residual connections, and a fused scan. Use a maintained implementation for performance-sensitive work; use the explicit loop to understand state semantics and masks.

### **Attention-SSM Hybrid Architectures** {#attention-ssm-hybrid-architectures}

Attention and SSMs provide complementary communication mechanisms:

- attention offers content-addressed pairwise interaction and direct retrieval;
- SSMs offer linear-time scanning, compressed recurrent state, and efficient streaming;
- local convolution offers short-range pattern extraction and stable high-throughput kernels.

A hybrid can alternate block types, place occasional attention layers among SSM layers, or split channels into parallel attention and state-space branches. The objective is not architectural decoration. Sparse attention layers can restore exact retrieval or global coordination while most layers retain linear sequence scaling.

Hybrids introduce interface questions. Branch outputs need compatible dimension and normalization. Causal masks must agree. If an attention branch uses unmasked padding while an SSM branch freezes padded state, their fusion leaks batch formatting. Streaming deployment must cache both recurrent SSM state and attention keys/values for the retained window.

The compute profile depends on frequency. If every block contains global attention, quadratic cost still dominates at long $T$. If attention appears every $m$ layers or uses local windows, the balance changes. Report the actual layer pattern and context window rather than labeling the model “linear” from one component.

<details>
<summary><strong>PyTorch: fuse masked attention and selective state on the same audio</strong></summary>

```python
class HybridSequenceBlock(nn.Module):
    def __init__(self, input_size=129, model_size=32):
        super().__init__()
        self.project = nn.Linear(input_size, model_size)
        self.attention = nn.MultiheadAttention(model_size // 2, num_heads=4, batch_first=True)
        self.ssm = SelectiveAcousticSSM(input_size=model_size // 2, state_size=12)
        self.fuse = nn.Linear(model_size, model_size)
        self.norm = nn.LayerNorm(model_size)

    def forward(self, x, mask):
        residual = self.project(x)
        attention_input, state_input = residual.chunk(2, dim=-1)
        attention_output, _ = self.attention(
            attention_input, attention_input, attention_input,
            key_padding_mask=~mask, need_weights=False,
        )
        state_scalar = self.ssm(state_input, mask)
        state_output = state_scalar.expand(-1, -1, state_input.shape[-1])
        mixed = self.fuse(torch.cat([attention_output, state_output], dim=-1))
        return self.norm(residual + mixed).masked_fill(~mask.unsqueeze(-1), 0.0)


hybrid = HybridSequenceBlock()
hybrid_output = hybrid(padded[:4], valid_mask[:4])
assert hybrid_output.shape == (4, padded.shape[1], 32)
assert torch.all(hybrid_output[~valid_mask[:4]] == 0)
print({"hybrid output": tuple(hybrid_output.shape),
       "valid vectors": int(valid_mask[:4].sum())})
```

</details>

The branch expansion is pedagogical rather than a published hybrid recipe. It makes the critical contract testable: both branches process the same valid time steps, fusion preserves shape, and padded outputs are explicitly removed.

### **RNNs, Transformers, and SSMs Compared** {#rnns-transformers-ssms-compared}

No sequence family is best independently of context length, latency, data scale, hardware, and the type of dependency the task requires.

| Family | Training interaction | Streaming state/cache | Long-range strength | Main limitation |
|---|---|---|---|---|
| vanilla RNN | sequential recurrence | fixed hidden state | compact but difficult to optimize | vanishing/exploding gradients and short bottleneck |
| LSTM/GRU | gated sequential recurrence | fixed recurrent state | stronger learned memory | limited parallelism and per-step gate cost |
| TCN | parallel local/dilated convolution | finite activation cache | controllable receptive field | finite context and possible dilation gaps |
| full Transformer | parallel all-pairs attention | KV cache growing with context | direct content-based retrieval | quadratic training interaction and cache growth |
| structured LTI SSM/S4 | parallel convolution or recurrent scan | fixed state | efficient long filters | content-independent base dynamics |
| selective SSM/Mamba | hardware-aware selective scan | fixed selective state | content-controlled compressed memory | no explicit pairwise retrieval and kernel dependence |
| hybrid | chosen mixture | both state and selected KV cache | balances retrieval and linear scanning | higher design and implementation complexity |

For the YESNO corpus, all methods receive the same normalized log-STFT frames and masks. That consistency reveals what actually changes: the state transition, dependency path, alignment objective, or communication operator. It also exposes what the dataset cannot answer. One speaker and sixty recordings cannot establish robustness across speakers or languages.

<details>
<summary><strong>PyTorch: verify a common sequence-model interface</strong></summary>

```python
interface_batch = padded[:3]
interface_lengths = lengths[:3]
interface_mask = valid_mask[:3]

rnn_logits = vanilla_rnn(interface_batch, interface_lengths)
lstm_logits = lstm_classifier(interface_batch, interface_lengths)
gru_logits = gru_classifier(interface_batch, interface_lengths)
ssm_features = selective_ssm(interface_batch, interface_mask)
hybrid_features = hybrid(interface_batch, interface_mask)

outputs = {
    "RNN logits": tuple(rnn_logits.shape),
    "LSTM logits": tuple(lstm_logits.shape),
    "GRU logits": tuple(gru_logits.shape),
    "selective SSM sequence": tuple(ssm_features.shape),
    "hybrid sequence": tuple(hybrid_features.shape),
}
assert rnn_logits.shape == lstm_logits.shape == gru_logits.shape == (3, 2)
assert ssm_features.shape[:2] == hybrid_features.shape[:2] == interface_batch.shape[:2]
print(outputs)
```

</details>

A fair predictive comparison would add equivalent heads, tune each family under a matched budget, repeat seeds, and report quality, peak memory, throughput, and streaming latency. Interface equality is necessary for that study but does not itself make capacities or optimization budgets equal.

### **Chapter Comparison and Summary** {#chapter-comparison-summary}

Sequence modeling is the design of an information path through ordered observations. The central questions are not only “which layer?” but also: what context is legally available, how padding is excluded, how long dependencies are optimized, how outputs align to inputs, and how state scales at training and inference.

| Mechanism | State update or interaction | Best-matched use | Diagnostic focus |
|---|---|---|---|
| vanilla recurrence | shared nonlinear transition | short streaming sequences and conceptual baseline | gradients through time and valid final state |
| LSTM | gated additive cell state | moderate/long streaming dependencies | gate saturation, state drift, recurrent latency |
| GRU | gated interpolation in one state | compact recurrent baseline | matched-width versus matched-budget comparison |
| bidirectional encoder | past and future recurrence | offline tagging/transcription | future-context latency violation |
| encoder-decoder | conditional autoregressive output | variable input/output lengths | bottleneck, teacher forcing, exposure bias |
| TCN | local/dilated temporal kernels | parallel finite-context processing | receptive-field coverage and causal padding |
| CTC | sum over monotonic alignments | unsegmented speech/handwriting | blank/repeat collapse and length constraints |
| S4-style SSM | structured linear state plus convolutional dual | long sequences and streaming | discretization stability and kernel generation |
| Mamba-style SSM | input-dependent selective scan | long content-sensitive streams | state capacity, masking, optimized kernels |
| hybrid | attention retrieval plus compressed state | mixed exact-recall and long-context tasks | branch masks, cache policy, actual complexity |

The chapter's continuous experiment follows this path:

1. Parse official waveforms, derive labels from filenames, and fit feature normalization on training frames only.
2. Pad variable-length batches with an explicit validity mask.
3. Use recurrent classifiers to expose state semantics and compare vanilla, LSTM, and GRU cells.
4. Unfold a real acoustic prefix to observe BPTT gradient behavior.
5. Preserve frame-level encoder outputs for bidirectionality, decoding, temporal convolution, and CTC.
6. Verify that a time-invariant SSM produces identical recurrent and convolutional outputs.
7. Make state transitions content dependent in a transparent selective scan.
8. Fuse attention and selective state only after aligning dimensions, masks, and causality.

The practical selection rule is conditional. Choose gated recurrence when fixed-state streaming and moderate sequences dominate; temporal convolution when a bounded receptive field and parallel training are natural; attention when direct content-based retrieval is essential; SSMs when long linear-time scanning and compact state are valuable; and hybrids when the task genuinely needs both retrieval and compressed memory. Then evaluate on representative lengths and hardware rather than inferring systems behavior from asymptotic notation alone.

Chapter 09 builds on these sequence foundations to study attention and Transformers in depth: query-key-value interaction, masking, positional information, normalization, feed-forward blocks, and efficient autoregressive inference.